# 1. Data Understanding

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import regexp_replace, col

## 1.1 Carga Inicial de Datos

In [0]:
dfAccesoInternetNoOficiales = spark.read.csv("/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/Instituciones-Educativas-no-Certificados-de-Boyacá-con-Acceso-a-Internet.csv",header=True, inferSchema=True)
dfAccesoInternetNoOficiales.createOrReplaceTempView("dfAccesoInternetNoOficiales")
display(dfAccesoInternetNoOficiales)

## 1.2 Descripción de Datos

In [0]:
dfAccesoInternetNoOficiales.printSchema()

In [0]:
df01 = dfAccesoInternetNoOficiales.withColumnRenamed("COD_DANE_IE", "codigo-dane-institucion-educativa") \
.withColumnRenamed("NOMBRE_IE", "institucion-educativa") \
.withColumnRenamed("COD_DANE_SEDE_IE", "codigo-dane-institucion-escuela") \
.withColumnRenamed("CÓDIGO MUNICIPIO", "codigo-municipio") \
.withColumnRenamed("COD_DANE_IE", "codigo-dane-institucion-educativa") \
.withColumnRenamed("NOMBRE_SEDE_IE", "escuela") \
.withColumnRenamed("MUNICIPIO", "municipio") \
.withColumnRenamed("LONGITUD", "longitud") \
.withColumnRenamed("LATITUD", "latitud") \
.withColumnRenamed("TIPO_SEDE", "tipo-sede") \
.withColumnRenamed("ZONA", "zona") \
.withColumnRenamed("TECNOLOGÍA_CONEXIÓN", "tecnologia-conexion") \
.withColumnRenamed("ANCHO_DE_BANDA(Mbps)", "ancho-de-banda") \
.withColumnRenamed("PROGRAMA_CONEXIÓN", "programa-conexion") \
.withColumnRenamed("AÑO", "anio") 


In [0]:
display(df01)

# 2. Exploración de Datos

In [0]:

#Primero se limpia el mbps de la columna de ancho de banda para poder hacer las operaciones bien hechas NOMBRAR ESTO EN EL DOC
df01 = df01.withColumn("ancho-de-banda-num",regexp_replace(col("ancho-de-banda"), " Mbps", "").cast("double"))

display(df01.describe())

Se cuenta con el registro de 409 instituciones educativas no oficiales en Boyacá, siendo menor que el de las oficiales (públicas). Al ser este dataset más pequeño y  del sector privado se espera que tengan mayor calidad en cuanto a la calidad y accesibilidad de internet. 

Respecto a la tecnología de conexión y ancho de banda hacen faltan 130 registros que pueden ser proyectos no registrados o colegios que simplemente no tienen alguno. 

Solamente se puede analizar el ancho de banda que hace falta el avg y la desviación estándar. Mientras que los estándares máximo y mínimo no están mal. 

Tocó utilizar Regex para pasar de de xx mbps a solamente el número. Así obteniendo que el ancho de banda está con un promedio de 13.27, que realmente está bajo para ser instituciones privadas con una desviación de 3.35. Por lo que hace falta calidad en el Internet. El mínimo y máximo siendo muy similar al de colegios públicos, aunque superior en el mínimo.

In [0]:
df02 = df01.groupBy(F.col("municipio")).count().orderBy("count", ascending=False).show(truncate=False)
display(df02)

La mayor cantidad de colegios no oficiales se encuentran en municipios de Tota, Paipa y Tibasosa. Municipios considerablemente cercanos a ciudades como Tunja o Bogotá. Aunque en general la tendencia se mantiene para todos los municipios. Se haya que Boyacá a pesar de componerse de 123 municipios, solamente encontramos esos, por lo que hacen falta: 

- Almeida  
- Arcabuco 
- Belén 
- Berbeo 
- Betéitiva 
- Boavita 
- Boyacá 
- Briceño 
- Buenavista 
- Busbanzá 
- Campohermoso 
- Cerinza 
- Chíquiza 
- Chiscas 
- Chita 
- Chitaraque 
- Chivatá 
- Chivor 
- Ciénega 
- Cómbita 
- Coper 
- Corrales 
- Covarachía 
- Cubará 
- Cucaita 
- Cuítiva 
- Duitama 
- El Cocuy 
- El Espino 
- Firvitoba 
- Floresta 
- Gachantivá 
- Gámeza 
- Garagoa 
- Guacamayas 
- Guateque 
- Guayatá 
- Guicán de la Sierra 
- Iza 
- Jenesano 
- Jericó
- La Capilla 
- La Uvita 
- La Victoria 
- Labranzagrande 
- Macanal 
- Maripí
- Miraflores 
- Monguí 
- Moniquirá 
- Motavita 
- Muzo  
- Nuevo Colón 
- Oicatá 
- Otanche 
- Pachavita 
- Páez  
- Pajarito 
- Panqueba 
- Pauna 
- Paya 
- Paz de Río 
- Pesca 
- Pisba  
- Quípama 
- Ramiriquí 
- Rondón 
- Sáchica 
- Samacá 
- San Eduardo 
- San José de Pare
- San Luis de Gaceno 
- San Mateo 
- San Miguel de Sema
- San Pablo de Borbur 
- Santa María
-  Santa Rosa de Viterbo 
-  Santa Sofía 
-  Santana  
-  Sativasur 
-  Siachoque 
-  Soatá 
-  Socha 
-  Socotá 
-  Somondoco 
-  Sora 
-  Soracá 
-  Sotaquirá 
-  Susacón 
-  Sutamarchán 
-  Sutatenza 
-  Tasco 
-  Tenza 
-  Tibaná 
-  Tibasosa 
-  Tinjaca 
-  Tipacoque 
-  Toca 
-  Togui 
-  Tópaga 
-  Tunja 
-  Tununguá 
-  Turmequé  
-  Tutazá  
-  Ventaquemada 
-  Villa de Leyva 
-  Viracachá

Lo que significa que estos municipios no tienen representación en este dataframe, para tener en cuenta. 

In [0]:
df03 = df01.groupBy(F.col("zona"),F.col("municipio")).count().orderBy("count", ascending=False).show(truncate=False)
display(df03)

La mayoría de colegios privados de este dataframe se encuentran en Tota, Paipa y Tibasosa. Y en general se tienen colegios rurales, no urbanos.

In [0]:

df04 = df01.groupBy("zona", "tecnologia-conexion").agg(F.avg("ancho-de-banda-num").alias("velocidad_promedio"),F.count("*").alias("cantidad"))

display(df04)

Databricks visualization. Run in Databricks to view.

Se obtienen resultados donde en las zonas urbanas se usa más fibra óptica, mientras que el las rurales se utiliza la radio. Resultado esperable por accesibilidad a zonas remotas.

In [0]:
df05 = df01.groupBy("programa-conexion").agg(F.avg("ancho-de-banda-num").alias("velocidad_promedio"),F.count("*").alias("cantidad")).orderBy(F.col("velocidad_promedio").desc())

display(df05)

Se observa como la los de CENTROS DIGITALES MINTIC están sin velocidad promedio, es decir, no figuran como comenzados en el dataframe. Por lo que la política a corte de 2023 no terminó por ser implementada.

In [0]:
df06 = df01.groupBy("tipo-sede").agg(F.avg("ancho-de-banda-num").alias("velocidad_promedio"),F.count("*").alias("cantidad"))

display(df06)

Se descubre que las sedes no principales mejoran un poco la velocidad promedio del Internet, sin embargo está bien tener en cuenta que hay menos sedes no principales que principales.

In [0]:
df07 = df01.groupBy("programa-conexion", "tecnologia-conexion").count()
display(df07)

Databricks visualization. Run in Databricks to view.

Como se refirió anteriormente, el programa de CENTROS DIGITALES no tiene ningún tipo de tecnología de conexión como no están iniciados. Apunte de que es una cantidad considerable de los centros educativos.

In [0]:
df08 = df01.groupBy("zona", "programa-conexion").agg(F.avg("ancho-de-banda-num").alias("velocidad_promedio"))
display(df08)

Databricks visualization. Run in Databricks to view.

In [0]:
Como el proyecto de CENTROS DIGITALES no cuenta con valores de velocidad, solamente se toma en cuenta la presencia del programa de Conexión Total que se ve sobretodo en colegios ubicados en zonas urbanas.

## 3. Reporte de Calidad de Datos

In [0]:
import pandas as pd

total = df01.count()

dfNulos = df01.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)for c in df01.columns])

nulos = dfNulos.collect()[0].asDict()

reporte = pd.DataFrame(list(nulos.items()), columns=["columna", "nulos"])
reporte["porcentaje"] = (reporte["nulos"] / total) * 100

display(reporte.sort_values(by="porcentaje", ascending=False))

In [0]:
df01.select(F.col("tecnologia-conexion")).distinct().show(truncate=False)
df01.select(F.col("ancho-de-banda")).distinct().show(truncate=False)



In [0]:
Se observa que los nulos solamente corresponden a tecnologia-conexion. Por este motivo se propone llenar los espacios como "Sin Datos" y no se busca algún tipo de promedio ya que no se tiene la información si el programa está por iniciar, no tiene o está cancelado. 

Mientras que también se propone eliminar la columna de ancho de banda ya que el valor es nulo y no se puede realizar ningún tipo de operación con el. La que realmente importa es ancho-de-banda-num.

Adicionalmente se crea la columna de tiene-ancho-banda en 1 si tiene ancho de banda y 0 si no tiene para comenzar a filtrar por los que tienen ancho de banda en relación al dataframe de colegios que si son públicos.

In [0]:
df01.select(F.col("programa-conexion")).distinct().show(truncate=False)
df01.select(F.col("anio")).distinct().show(truncate=False)

In [0]:
Se observan que tienen solamente 2 programas activos para dar Internet a colegios privados, ya que probablemente la mayoría ya tengan Internet incluido o no estén en el dataset. No se considera necesario hacer modificaciones.

## 4. Filtros y Limpieza

### 4.1 Eliminar columna de ancho-de-banda no numérico

In [0]:
if df01 is not None: df01 = df01.drop("ancho-de-banda")

### 4.2 Rellenar con SIN DATOS Y poner en Mayúscula

In [0]:
df01 = df01.fillna({"tecnologia-conexion": "SIN DATOS"})

In [0]:
df01 = df01.withColumn("tecnologia-conexion",F.upper(F.trim(F.col("tecnologia-conexion"))))

### 4.3 Creación de nueva columna

In [0]:
df01 = df01.withColumn("tiene-ancho-banda",F.when(F.col("ancho-de-banda-num").isNotNull(), 1).otherwise(0))

In [0]:
display(df01)